# Exploring the Sequential Matching data

Notes from a first pass through `data/public_01 ... public_10`. Everything here is synthetic.
The questions I want answered before choosing an approach:

1. What does a member record look like, and how much of it is missing?
2. How many valid (feasible) pairs exist, and how many are blocked only by missing answers?
3. What do the historical introductions and feedback look like, and how many positive outcomes are there?
4. What can and can't be used as a training label?

Run this notebook from the `exploration/` folder (or the repo root). It only reads files in `data/` and imports `kit.py` for the eligibility rules.

In [ ]:
import json, sys, collections, itertools
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "kit.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import kit

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
print("repo root:", ROOT)
print("hard fields:", kit.HARD)
print("soft fields:", kit.SOFT)

## 1. Manifest and splits

The manifest tells us which pools are for training, validation and development testing. Rows from the same people must never be split across these.

In [ ]:
manifest = json.load(open(ROOT / "data_manifest.json"))
split_of = {}
for split_name, pool_ids in manifest["splits"].items():
    for pid in pool_ids:
        split_of[pid] = split_name

pools_df = pd.DataFrame(manifest["pools"])
pools_df["split"] = pools_df["pool_id"].map(split_of)
pools_df

In [ ]:
print(pools_df.groupby("split")[["members", "introductions", "feedback_events"]].sum())
print()
print("totals:", pools_df[["members", "introductions", "feedback_events"]].sum().to_dict())

## 2. Loading everything

I load each pool's files into plain lists first, then build DataFrames. `members.jsonl` has the per-field status and observation day, which `members_flat.csv` drops, so I use the jsonl.

In [ ]:
states = {}
member_rows = []
intro_rows = []
fb_rows = []
conv_rows = []

for pool_dir in sorted((ROOT / "data").glob("public_*")):
    pid = pool_dir.name
    states[pid] = json.load(open(pool_dir / "state.json"))

    for line in open(pool_dir / "members.jsonl", encoding="utf-8"):
        m = json.loads(line)
        row = {
            "member_id": m["member_id"], "pool_id": m["pool_id"], "age": m["age"],
            "gender": m["gender"], "zone": m["zone"], "arrived_day": m["arrived_day"],
            "available": m["available"],
        }
        for k, v in m["fields"].items():
            row["f_" + k] = v
        for k, v in m["field_status"].items():
            row["s_" + k] = v
        for k, v in m["field_observed_day"].items():
            row["d_" + k] = v
        member_rows.append(row)

    for line in open(pool_dir / "introductions.jsonl", encoding="utf-8"):
        r = json.loads(line)
        r["pool_id"] = pid
        intro_rows.append(r)

    for line in open(pool_dir / "feedback.jsonl", encoding="utf-8"):
        r = json.loads(line)
        r["pool_id"] = pid
        fb_rows.append(r)

    for line in open(pool_dir / "conversations.jsonl", encoding="utf-8"):
        r = json.loads(line)
        r["pool_id"] = pid
        conv_rows.append(r)

members = pd.DataFrame(member_rows)
intros = pd.DataFrame(intro_rows)
feedback = pd.DataFrame(fb_rows)
print("members", members.shape, "| introductions", intros.shape, "| feedback", feedback.shape, "| conversations", len(conv_rows))

In [ ]:
# does state.json agree with members.jsonl?
for pid, st in states.items():
    n_state = len(st["members"])
    n_file = (members["pool_id"] == pid).sum()
    assert n_state == n_file, (pid, n_state, n_file)
    assert st["day"] == 30
print("state.json and members.jsonl agree; every snapshot is day", {st["day"] for st in states.values()})
print("ask budget remaining in snapshots:", {st["ask_budget_remaining"] for st in states.values()})

## 3. What a member looks like

In [ ]:
members.iloc[:3].T

In [ ]:
members.dtypes.value_counts()

## 4. Who is in the pools

Age, gender, zone and arrival day. Arrival day matters: people keep appearing up to day 20, so a policy can't see the whole pool on day 0.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

members["age"].plot.hist(bins=range(20, 49), ax=axes[0, 0], edgecolor="white")
axes[0, 0].set_title("age")

members["gender"].value_counts().plot.bar(ax=axes[0, 1])
axes[0, 1].set_title("gender")

members["zone"].value_counts().sort_index().plot.bar(ax=axes[1, 0])
axes[1, 0].set_title("home zone")

members["arrived_day"].value_counts().sort_index().plot.bar(ax=axes[1, 1])
axes[1, 1].set_title("arrival day")

plt.tight_layout()
plt.show()

In [ ]:
print(members["gender"].value_counts(normalize=True).round(3))
print()
print(members["zone"].value_counts(normalize=True).sort_index().round(3))
print()
print("share arrived on day 0:", round((members["arrived_day"] == 0).mean(), 3))
print("share currently unavailable at day 30:", round((~members["available"]).mean(), 3))

In [ ]:
# gender balance per pool: matching needs people who want to meet each other
pd.crosstab(members["pool_id"], members["gender"])

## 5. Missing data

Every field has a status: `observed`, `not_asked` or `declined`. A null value is unknown, never zero. First, how many of each status per field?

In [ ]:
status_cols = [c for c in members.columns if c.startswith("s_")]
long_status = members[status_cols].melt(var_name="field", value_name="status")
long_status["field"] = long_status["field"].str.replace("s_", "", regex=False)
status_table = pd.crosstab(long_status["field"], long_status["status"])
status_table["pct_not_observed"] = (1 - status_table["observed"] / status_table.sum(axis=1)).round(3)
status_table.sort_values("pct_not_observed", ascending=False)

In [ ]:
share = status_table.drop(columns="pct_not_observed")
share = share.div(share.sum(axis=1), axis=0).sort_values("observed")
share.plot.barh(stacked=True, figsize=(9, 6))
plt.title("field status share (all pools)")
plt.xlabel("share of members")
plt.show()

In [ ]:
# the 11 hard fields are what block introductions, so look at them on their own
hard_status_cols = ["s_" + k for k in kit.HARD]
hard_val_cols = ["f_" + k for k in kit.HARD]

members["n_hard_missing"] = members[hard_val_cols].isna().sum(axis=1)
members["n_hard_declined"] = (members[hard_status_cols] == "declined").sum(axis=1)

print("members by number of missing hard fields:")
print(members["n_hard_missing"].value_counts().sort_index())
print()
print("members with at least one declined hard field:", int((members["n_hard_declined"] > 0).sum()))

In [ ]:
# a declined answer stays unknown even after an ask, so a member with a declined hard field can never become feasible.
# how do the missing-hard-data members split into declined vs merely not-asked?
has_missing = members["n_hard_missing"] > 0
has_declined = members["n_hard_declined"] > 0
print(pd.crosstab(has_missing, has_declined, rownames=["missing hard field"], colnames=["declined hard field"]))
print()
print("members permanently blocked (declined a hard field):", int(has_declined.sum()), "of", len(members))
print("share of all members:", round(has_declined.mean(), 3))

Every member with missing hard data has declined at least one hard field. Asking cannot recover a declined answer, so these members can never be introduced to anyone under the contract. That is a hard ceiling on coverage, before any modelling.

In [ ]:
# is missingness tied to arrival day, or to the pool?
print(members.groupby("arrived_day")["n_hard_missing"].mean().round(2).to_string())
print()
print(members.groupby("pool_id")["n_hard_missing"].agg(["mean", "max"]).round(2))

In [ ]:
# when a value is null, does the status always say why? any observed value with a null, or vice versa?
bad = 0
for k in kit.HARD + kit.SOFT:
    is_null = members["f_" + k].isna()
    is_obs = members["s_" + k] == "observed"
    bad += int((is_null & is_obs).sum()) + int((~is_null & ~is_obs).sum())
print("rows where null and status disagree:", bad)

## 6. Stated preferences (hard fields)

In [ ]:
members["age_range_width"] = members["f_age_max"] - members["f_age_min"]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
members["f_age_min"].plot.hist(bins=20, ax=axes[0], edgecolor="white"); axes[0].set_title("age_min")
members["f_age_max"].plot.hist(bins=20, ax=axes[1], edgecolor="white"); axes[1].set_title("age_max")
members["age_range_width"].plot.hist(bins=20, ax=axes[2], edgecolor="white"); axes[2].set_title("age range width")
plt.tight_layout()
plt.show()

In [ ]:
members["who_str"] = [",".join(v) if isinstance(v, list) else None for v in members["f_who_to_meet"]]
print(pd.crosstab(members["gender"], members["who_str"].fillna("unknown")))

In [ ]:
for k in ["relationship_structure", "smoking", "partner_smoking", "has_children", "partner_children", "wants_children"]:
    print(k)
    print(members["f_" + k].value_counts(dropna=False).to_string())
    print()

In [ ]:
# acceptable zones and schedules are lists
members["n_zones"] = [len(v) if isinstance(v, list) else None for v in members["f_acceptable_zones"]]
members["n_slots"] = [len(v) if isinstance(v, list) else None for v in members["f_schedule"]]
members["accepts_own_zone"] = [
    (z in v) if isinstance(v, list) else None for z, v in zip(members["zone"], members["f_acceptable_zones"])
]

print("number of acceptable zones:")
print(members["n_zones"].value_counts(dropna=False).sort_index().to_string())
print()
print("number of schedule slots:")
print(members["n_slots"].value_counts(dropna=False).sort_index().to_string())
print()
print("accepts own zone:")
print(members["accepts_own_zone"].value_counts(dropna=False).to_string())

In [ ]:
slot_counts = collections.Counter()
for v in members["f_schedule"]:
    if isinstance(v, list):
        slot_counts.update(v)
print(slot_counts)

## 7. Softer fields

These never block a pair. They are the only things the baseline uses to rank pairs.

In [ ]:
for k in kit.SOFT:
    print(k)
    print(members["f_" + k].value_counts(dropna=False).to_string())
    print()

In [ ]:
# are soft traits related to each other or to gender / age?
print(pd.crosstab(members["f_relationship_goal"], members["f_relationship_pace"]))
print()
print(pd.crosstab(members["gender"], members["f_relationship_goal"], normalize="index").round(3))
print()
print(members.groupby("f_relationship_goal")["age"].mean().round(2))

## 8. Questionnaires and conversations

`questionnaires.jsonl` repeats the same answers as text and `conversations.jsonl` is a short scripted chat. Quick check that they add nothing new.

In [ ]:
q_rows = []
for pool_dir in sorted((ROOT / "data").glob("public_*")):
    for line in open(pool_dir / "questionnaires.jsonl", encoding="utf-8"):
        q_rows.append(json.loads(line))
print("questionnaires:", len(q_rows))
print(json.dumps(q_rows[0]["questionnaire_answers"], indent=1))

In [ ]:
conv_df = pd.DataFrame(conv_rows)
print("conversations:", len(conv_df), "of", len(members), "members")
print("observed_day values:", conv_df["observed_day"].unique())
print("styles:", conv_df["style"].unique())

user_msgs = collections.Counter()
for msgs in conv_df["messages"]:
    for m in msgs:
        if m["role"] == "user":
            user_msgs[m["content"]] += 1
print("distinct user replies:", len(user_msgs))
for text, n in user_msgs.most_common(12):
    print(n, "|", text)

The distinct user replies are a small closed set, so the conversations are templates that map one-to-one onto the four soft fields. They are an input-handling exercise, not extra signal.

## 9. Who can meet whom: the feasibility graph

I use the supplied `kit.eligibility(a, b)` on every pair inside each pool, using the day-30 snapshot. Note that `eligibility` ignores availability and past introductions.

In [ ]:
edge_rows = []
for pid, st in states.items():
    ms = st["members"]
    for a, b in itertools.combinations(ms, 2):
        res = kit.eligibility(a, b)
        edge_rows.append({
            "pool_id": pid, "a": a["member_id"], "b": b["member_id"],
            "status": res["status"], "reasons": res["reasons"], "missing": res["missing"],
        })
edges = pd.DataFrame(edge_rows)
print("pairs evaluated:", len(edges))
edges["status"].value_counts()

In [ ]:
status_share = edges["status"].value_counts(normalize=True).round(4)
print(status_share)
print()
print(pd.crosstab(edges["pool_id"], edges["status"]))

In [ ]:
# why are pairs infeasible? a pair can fail for several reasons at once
reason_counter = collections.Counter()
for r in edges.loc[edges["status"] == "infeasible", "reasons"]:
    reason_counter.update(r)
reason_df = pd.Series(reason_counter).sort_values(ascending=False)
print(reason_df)
reason_df.plot.bar(title="reasons a pair is infeasible (a pair can have several)")
plt.show()

In [ ]:
# for pairs needing clarification: which fields are missing, and how many?
missing_field_counter = collections.Counter()
for r in edges.loc[edges["status"] == "needs_clarification", "missing"]:
    missing_field_counter.update([x.split(":")[1] for x in r])
print(pd.Series(missing_field_counter).sort_values(ascending=False))
print()
nc = edges[edges["status"] == "needs_clarification"]
print("distinct members involved in clarification-blocked pairs:", len(set(nc["a"]) | set(nc["b"])))

Now the same graph from the members' side: how many partners does each person have?

In [ ]:
feasible_deg = collections.Counter()
nc_deg = collections.Counter()
for a, b in zip(edges.loc[edges["status"] == "feasible", "a"], edges.loc[edges["status"] == "feasible", "b"]):
    feasible_deg[a] += 1
    feasible_deg[b] += 1
for a, b in zip(nc["a"], nc["b"]):
    nc_deg[a] += 1
    nc_deg[b] += 1

members["n_feasible"] = members["member_id"].map(feasible_deg).fillna(0).astype(int)
members["n_needs_clar"] = members["member_id"].map(nc_deg).fillna(0).astype(int)
members["n_potential"] = members["n_feasible"] + members["n_needs_clar"]

print(members[["n_feasible", "n_needs_clar", "n_potential"]].describe().round(2))
print()
print("members with no feasible partner now:", int((members["n_feasible"] == 0).sum()))
print("members with no feasible AND no potential partner:", int((members["n_potential"] == 0).sum()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
members["n_feasible"].plot.hist(bins=30, ax=axes[0], edgecolor="white")
axes[0].set_title("feasible partners per member (day 30)")
members["n_potential"].plot.hist(bins=30, ax=axes[1], edgecolor="white")
axes[1].set_title("feasible + needs-clarification partners")
plt.tight_layout()
plt.show()

In [ ]:
# who has no partner at all? compare them with everyone else
members["isolated"] = members["n_potential"] == 0
print(members.groupby("isolated")[["age", "n_hard_missing"]].mean().round(2))
print()
print(pd.crosstab(members["isolated"], members["gender"]))
print()
print(pd.crosstab(members["isolated"], members["f_relationship_structure"].fillna("unknown")))

In [ ]:
# degree by gender / structure: the supply side of the problem
print(members.groupby("gender")["n_feasible"].mean().round(2))
print()
print(members.groupby(members["f_relationship_structure"].fillna("unknown"))["n_feasible"].mean().round(2))

## 10. Historical introductions

Who was introduced to whom, and when? Remember these come from a baseline-style policy, so they are not a random sample of pairs.

In [ ]:
intros.head()

In [ ]:
print(intros.groupby("pool_id").size().to_string())
print()
print("logging policies:", intros["logging_policy"].unique(), "| propensity values:", intros["propensity"].unique())
print("response window (deadline - assigned):", (intros["response_deadline_day"] - intros["assigned_day"]).unique())

In [ ]:
intros["assigned_day"].value_counts().sort_index().plot.bar(figsize=(11, 3.5), title="introductions assigned per day")
plt.show()

In [ ]:
# how many times was each member introduced? any repeated pair?
appearances = collections.Counter()
pair_counter = collections.Counter()
for pid, a, b in zip(intros["pool_id"], intros["user_a"], intros["user_b"]):
    appearances[a] += 1
    appearances[b] += 1
    pair_counter[(pid, tuple(sorted((a, b))))] += 1

members["n_intros"] = members["member_id"].map(appearances).fillna(0).astype(int)
print("introductions per member:")
print(members["n_intros"].value_counts().sort_index().to_string())
print()
print("pairs introduced more than once:", sum(1 for v in pair_counter.values() if v > 1))
print("coverage so far (introduced at least once):", round((members["n_intros"] > 0).mean(), 3))

In [ ]:
# were the historical pairs valid under today's eligibility? (they should be, if the fields were known then)
status_lookup = {}
for pid, a, b, s in zip(edges["pool_id"], edges["a"], edges["b"], edges["status"]):
    status_lookup[(pid, tuple(sorted((a, b))))] = s

intros["elig_now"] = [status_lookup[(pid, tuple(sorted((a, b))))]
                      for pid, a, b in zip(intros["pool_id"], intros["user_a"], intros["user_b"])]
print(intros["elig_now"].value_counts())

In [ ]:
# did the older policy introduce people before their hard fields were visible?
# compare assigned_day with the day each endpoint's hard fields became observable
member_index = members.set_index("member_id")
late_info = []
for r in intro_rows:
    late = 0
    for uid in (r["user_a"], r["user_b"]):
        days = [member_index.loc[uid, "d_" + k] for k in kit.HARD]
        late += sum(1 for d in days if pd.notna(d) and d > r["assigned_day"])
    late_info.append(late)
intros["hard_fields_learned_after_intro"] = late_info
print("introductions where some hard field was only observed AFTER the assignment:",
      int((intros["hard_fields_learned_after_intro"] > 0).sum()), "of", len(intros))

That last number matters: if the day-30 profile contains answers learned after the introduction, then using the final profile as features for an earlier outcome leaks information the policy didn't have.

## 11. Feedback events

In [ ]:
feedback.head(8)

In [ ]:
print(feedback["event"].value_counts())
print()
print(pd.crosstab(feedback["event"], feedback["value"].astype(str)))
print()
print(feedback["missing_reason"].value_counts(dropna=False))

In [ ]:
# how fast do people answer? latency = observed_day - assigned_day
resp = feedback[feedback["event"] == "introduction_response"].merge(
    intros[["introduction_id", "assigned_day", "response_deadline_day"]], on="introduction_id")
resp["latency"] = resp["observed_day"] - resp["assigned_day"]

print(resp.groupby(resp["value"].fillna("no_response"))["latency"].describe().round(2))
resp[resp["value"].notna()]["latency"].value_counts().sort_index().plot.bar(title="days to answer (when they answered)")
plt.show()

### Per-introduction outcome and the funnel

I build one row per introduction. The status labels are my own reading of the statement's rules, so treat them as exploratory:

- **success**: date happened and both said yes to meeting again within 3 days of the date (that is MSMI).
- **failed**: someone said no, or the deadline passed without two yes answers, or no date happened, or someone declined a second meeting.
- **open**: nothing has ruled it out yet, but the day-30 snapshot is too early to know (right-censored).

In [ ]:
fb_by_intro = collections.defaultdict(list)
for r in fb_rows:
    fb_by_intro[r["introduction_id"]].append(r)

outcome_rows = []
for r in intro_rows:
    es = fb_by_intro[r["introduction_id"]]
    answers = {e["member_id"]: e["value"] for e in es if e["event"] == "introduction_response"}
    n_yes = sum(v == "yes" for v in answers.values())
    n_no = sum(v == "no" for v in answers.values())
    n_silent = sum(v is None for v in answers.values())

    date_true = [e for e in es if e["event"] == "date_happened" and e["value"] is True]
    date_false = [e for e in es if e["event"] == "date_happened" and e["value"] is False]
    date_day = date_true[0]["occurred_day"] if date_true else None

    second = [e for e in es if e["event"] == "second_meeting_intention"]
    sm_yes_in_window = 0
    sm_no = 0
    for e in second:
        if e["value"] == "no":
            sm_no += 1
        if e["value"] == "yes" and date_day is not None and e["occurred_day"] - date_day <= 3:
            sm_yes_in_window += 1

    msmi = sm_yes_in_window == 2
    deadline_passed = r["response_deadline_day"] <= 30
    if msmi:
        status = "success"
    elif n_no > 0 or sm_no > 0 or date_false or (deadline_passed and n_yes < 2):
        status = "failed"
    else:
        status = "open"

    outcome_rows.append({
        "introduction_id": r["introduction_id"], "pool_id": r["pool_id"], "assigned_day": r["assigned_day"],
        "user_a": r["user_a"], "user_b": r["user_b"],
        "n_yes": n_yes, "n_no": n_no, "n_silent": n_silent,
        "mutual_accept": n_yes == 2, "date": bool(date_true), "date_day": date_day,
        "n_second_yes_in_window": sm_yes_in_window, "msmi": msmi, "status": status,
    })

outcomes = pd.DataFrame(outcome_rows)
outcomes["status"].value_counts()

In [ ]:
funnel = pd.Series({
    "introduced": len(outcomes),
    "at least one yes": int((outcomes["n_yes"] >= 1).sum()),
    "mutual acceptance": int(outcomes["mutual_accept"].sum()),
    "date happened": int(outcomes["date"].sum()),
    "MSMI": int(outcomes["msmi"].sum()),
})
print(funnel)
print()
print((funnel / funnel.iloc[0] * 100).round(1).astype(str) + " % of introductions")
funnel.plot.bar(title="outcome funnel, all 10 pools, day 30")
plt.show()

In [ ]:
# response behaviour: yes / no / silent, per person-response
resp_counts = feedback[feedback["event"] == "introduction_response"]["value"].fillna("no_response").value_counts()
print(resp_counts)
print((resp_counts / resp_counts.sum()).round(3))

In [ ]:
# how many MSMI per pool, and per split?
outcomes["split"] = outcomes["pool_id"].map(split_of)
print(outcomes.groupby("pool_id")[["mutual_accept", "date", "msmi"]].sum())
print()
print(outcomes.groupby("split")[["mutual_accept", "date", "msmi"]].sum())

In [ ]:
# censoring: how many introductions are too recent to have finished?
outcomes["response_window_open"] = (outcomes["assigned_day"] + 7) > 30
outcomes["date_window_open"] = (outcomes["assigned_day"] + 30) > 30
print("response window still open at day 30:", int(outcomes["response_window_open"].sum()))
print("30-day date window still open at day 30:", int(outcomes["date_window_open"].sum()), "of", len(outcomes))
print()
print(pd.crosstab(outcomes["response_window_open"], outcomes["status"]))

Every introduction after day 0 still has an open 30-day date window at the day-30 snapshot, so this snapshot can only give a lower bound on MSMI. That is why the statement insists on mature outcomes after the 40-day follow-up.

## 12. Does soft-trait similarity relate to the outcome?

The baseline ranks pairs by how many soft fields match. Is there any sign in the history that this helps? The caveat is large: these are only pairs the old policy chose, and the sample is small.

In [ ]:
sim_scores = []
for a, b in zip(outcomes["user_a"], outcomes["user_b"]):
    n_same = 0
    for k in kit.SOFT:
        va = member_index.loc[a, "f_" + k]
        vb = member_index.loc[b, "f_" + k]
        if pd.notna(va) and pd.notna(vb) and va == vb:
            n_same += 1
    sim_scores.append(n_same)
outcomes["soft_matches"] = sim_scores

by_sim = outcomes.groupby("soft_matches").agg(
    n=("introduction_id", "count"),
    yes_total=("n_yes", "sum"),
    mutual=("mutual_accept", "sum"),
    dates=("date", "sum"),
    msmi=("msmi", "sum"),
)
by_sim["mutual_rate"] = (by_sim["mutual"] / by_sim["n"]).round(3)
by_sim["yes_rate"] = (by_sim["yes_total"] / (2 * by_sim["n"])).round(3)
by_sim

In [ ]:
print("number of soft fields:", len(kit.SOFT))
print("correlation of soft_matches with mutual acceptance:", round(outcomes["soft_matches"].corr(outcomes["mutual_accept"].astype(float)), 3))
print("correlation of soft_matches with number of yes answers:", round(outcomes["soft_matches"].corr(outcomes["n_yes"].astype(float)), 3))

In [ ]:
# individual traits: does the yes rate differ by the responder's own traits?
resp_traits = feedback[feedback["event"] == "introduction_response"].merge(
    members[["member_id", "age", "gender", "zone", "f_relationship_goal", "f_relationship_pace",
             "f_lifestyle", "f_conversations", "f_emotional_availability", "f_space_for_relationship"]],
    on="member_id", how="left")
resp_traits["said_yes"] = (resp_traits["value"] == "yes").astype(int)
resp_traits["answered"] = resp_traits["value"].notna().astype(int)

for col in ["gender", "f_relationship_goal", "f_relationship_pace", "f_lifestyle",
            "f_emotional_availability", "f_space_for_relationship"]:
    t = resp_traits.groupby(col, dropna=False).agg(n=("said_yes", "size"),
                                                   answered=("answered", "mean"),
                                                   yes_rate=("said_yes", "mean")).round(3)
    print(col)
    print(t.to_string())
    print()

## 13. Clarification (ask) history

In [ ]:
ask_rows = []
for pid, st in states.items():
    for a in st["ask_log"]:
        ask_rows.append({"pool_id": pid, "member_id": a["member_id"], "field": a["field"],
                         "observed_day": a["observed_day"],
                         "cost": 3 if a["field"] == "constraints" else 1})
asks = pd.DataFrame(ask_rows)
print("asks:", len(asks), "| fields asked:", asks["field"].value_counts().to_dict())
print("total cost:", int(asks["cost"].sum()), "| cost per pool:", asks.groupby("pool_id")["cost"].sum().to_dict())

In [ ]:
daily = asks.groupby(["pool_id", "observed_day"])["cost"].sum()
print("max spend in one day:", int(daily.max()), "(budget is 12)")
asks.groupby("observed_day")["cost"].sum().plot.bar(figsize=(11, 3.5), title="clarification units spent per day (all pools)")
plt.show()

In [ ]:
# were the asks aimed at the right people? compare asked members with members still missing hard data at day 30
asked_ids = set(asks.loc[asks["field"] == "constraints", "member_id"])
members["was_asked_constraints"] = members["member_id"].isin(asked_ids)
print(pd.crosstab(members["was_asked_constraints"], members["n_hard_missing"] > 0,
                  rownames=["asked"], colnames=["still missing hard data"]))
print()
print(members.groupby("was_asked_constraints")[["n_feasible", "n_potential", "arrived_day"]].mean().round(2))

## 14. What I take away

What the numbers above showed (day-30 snapshots, 10 pools, 2,000 members):

- **A third of members are unreachable.** 700 members (35%) have declined at least one hard field. A declined answer stays unknown, so they can never be introduced. Coverage cannot exceed about 65% whatever the policy does.
- **Feasible pairs are rare.** Of 199,000 pairs, about 0.5% are feasible now, 12.9% need clarification and 86.6% are infeasible. Most "needs clarification" pairs involve a declined member and can never resolve, so check how many can actually be unlocked before spending budget.
- **Asking is only useful for the not-asked members.** The history shows 630 hard-constraint asks (1,890 units), all of which resolved their member completely.
- **Labels are scarce.** 613 introductions gave 419 yes answers, 84 mutual acceptances, 40 dates and only 3 MSMI. Training directly on MSMI is not possible. Individual yes answers (419 yes, 467 no, 249 no response) are the realistic target.
- **Soft similarity is a weak signal.** The number of matching soft fields correlates about 0.1 with mutual acceptance in this selected sample. Treat it as unproven.
- **Bias and censoring.** Outcomes exist only for pairs the earlier policy chose, propensities are null, and every introduction after day 0 still has an open 30-day date window. The day-30 profile did not leak later hard answers into these introductions (0 of 613), but check this again for soft fields before using them as features.
- **Timing.** Members arrive up to day 20 (70% on day 0), and answers take about 4 days on average, so memory and waiting matter.

Next steps: run the baselines on several simulator seeds to get more labelled rollouts, then compare global matching and targeted asking against the greedy baseline.